# Performance Relation

分析原始数据序列与创造性绩效的关系。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 输入文件路径
output_dir = Path('output')
raw_sequences_file = output_dir / 'raw_sequences.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'

In [ ]:
# 加载数据
df_raw_sequences = pd.read_csv(raw_sequences_file)
df_performance = pd.read_csv(performance_file)

In [ ]:
# 1. 计算原始序列余弦相似性矩阵
from sklearn.metrics.pairwise import cosine_similarity
import ast
import json

def cosine_similarity_matrix(df_sequences):
    # 解析字符串形式的数组，将其转换回numpy array
    def parse_sequence(seq_str):
        if not isinstance(seq_str, str):
            return seq_str
        try:
            return json.loads(seq_str)
        except:
            pass
        try:
            return ast.literal_eval(seq_str)
        except:
            # 备用方案：处理可能是无逗号空格分隔的数组字符串
            clean_str = seq_str.strip('[]\n\r ')
            if ',' in clean_str:
                return [float(x) for x in clean_str.split(',')]
            return [float(x) for x in clean_str.split()]
            
    # 解析并构成特征矩阵 X
    parsed_seqs = df_sequences['raw_sequence'].apply(parse_sequence).tolist()
    X = np.array(parsed_seqs)
    
    # 用sklearn计算两两序列的余弦相似度矩阵
    cos_sim_matrix = cosine_similarity(X)
    return cos_sim_matrix

In [ ]:
# 2. 计算表现之间的差值的绝对值矩阵

def performance_difference_matrix(performance_df, key):
    # 提取表现得分
    perf_scores = performance_df[key].values
    
    # 利用numpy的广播机制，直接计算两两之间分数的差值绝对值
    # perf_scores[:, None] 将 (N,) 变成 (N, 1)
    # perf_scores[None, :] 变成 (1, N)
    diff_matrix = np.abs(perf_scores[:, None] - perf_scores[None, :])
    
    return diff_matrix

In [ ]:
# 3. 计算两个矩阵之间的相关性
from scipy.stats import pearsonr, spearmanr

def calculate_correlation(matrix1, matrix2, method='spearman'):
    # 取矩阵的上三角（不含对角线 k=1）的部分进行相关性计算
    # 这样可以避免自身与自身的相关（对角线），以及对称性的冗余（双倍计算）
    indices = np.triu_indices_from(matrix1, k=1)
    
    vec1 = matrix1[indices]
    vec2 = matrix2[indices]
    
    # 计算相关系数，通常用Spearman评估非线性单调关系，因为表现差距不一定是线性的
    if method == 'pearson':
        corr, pval = pearsonr(vec1, vec2)
    else:
        corr, pval = spearmanr(vec1, vec2)
        
    return corr

In [ ]:
# 4. 验证相关性的可靠性，进行以下迭代1000~10000次：
# (1) 随机打乱原创性矩阵的数据点分布
# (2) 重新计算相关性
# (3) 记录每次迭代的相关性值，形成一个分布
# (4) 计算原始相关性在这个分布中的位置，评估其显著性
from tqdm import tqdm

def shuffle_matrix(matrix):
    # Mantel test要求：在打乱矩阵时，必须同步打乱行和列的索引。
    # 这样才能保持矩阵内部的结构逻辑假说被打破，同时仍保留原有的节点度分布。
    n = matrix.shape[0]
    idx = np.random.permutation(n)
    # matrix[idx, :] 会打乱行，接着 [:, idx] 会按照同样的顺序打乱列
    return matrix[idx, :][:, idx]

def permutation_test(cosine_matrix, originality_matrix, num_iterations=1000, method='spearman'):
    # 首先计算实际观察到的真实相关性
    true_corr = calculate_correlation(cosine_matrix, originality_matrix, method=method)
    
    null_distribution = np.zeros(num_iterations)
    
    # 不断打乱和重算，构造零假设分布
    for i in tqdm(range(num_iterations), desc="Permutation Testing"):
        shuffled_orig = shuffle_matrix(originality_matrix)
        null_distribution[i] = calculate_correlation(cosine_matrix, shuffled_orig, method=method)
        
    # 计算双尾回归 P 值：记录比观察到的真实相关性“更极端”的数据占比
    p_value = np.sum(np.abs(null_distribution) >= np.abs(true_corr)) / num_iterations
    
    return true_corr, p_value, null_distribution

In [ ]:
# 主流程

# 计算余弦相似性矩阵
cosine_matrix = cosine_similarity_matrix(df_raw_sequences)

# 计算表现差值矩阵
performance_matrix = performance_difference_matrix(df_performance, 'fluency')

# 计算相关性
correlation, p_value, null_dist = permutation_test(cosine_matrix, performance_matrix, num_iterations=1000, method='spearman')

print(f"Observed Spearman Correlation: {correlation:.4f}")
print(f"P-value from Permutation Test: {p_value:.4f}")

In [ ]:
# 可视化零假设分布和观察到的相关性
plt.figure(figsize=(10, 6))
plt.hist(null_dist, bins=50, alpha=0.7, label='Null Distribution')
plt.axvline(correlation, color='red', linestyle='--', linewidth=2, label=f'Observed Correlation: {correlation:.4f}')
plt.xlabel('Spearman Correlation')
plt.ylabel('Frequency')
plt.title('Permutation Test for Performance Relation')
plt.legend()
plt.show()